# Introduction to RAG Systems

## 1. The Problem: LLMs Don't Know Your Data

Large Language Models like GPT-4 are trained on a massive, static snapshot of the public internet. This leads to two major limitations:
1.  **Knowledge Cutoff:** They have no information about events that occurred after their training date.
2.  **Lack of Private Context:** They do not know about your company's internal documents, your personal notes, or any other private data source.

**RAG solves this.** It is a technique for providing an LLM with external knowledge it wasn't trained on, grounding its answers in facts and reducing hallucinations.

### Real-World Case Study: Customer Support Automation

A large e-commerce company implemented a RAG system to enhance their customer support chatbot. The chatbot was trained on a vast array of internal documents, including product manuals, customer service guidelines, and past customer queries. By integrating RAG, the chatbot could provide accurate and context-aware responses to customer inquiries, significantly improving customer satisfaction and reducing the need for human intervention.

## 2. The RAG Workflow: How It Works

The RAG pipeline has two main stages: **Indexing** (a one-time, offline process) and **Retrieval & Generation** (the real-time process).

### Stage 1: Indexing (Preparing the Knowledge Base)

1.  **Load Documents:** Your raw data (e.g., PDFs, Markdown files, Confluence pages) is loaded.
2.  **Chunking:** The documents are split into smaller, manageable chunks. This is crucial because LLMs have a limited context window.
3.  **Embedding:** Each chunk is passed through an **embedding model** (like `all-MiniLM-L6-v2`), which converts the text into a numerical vector. This vector captures the semantic meaning of the text.
4.  **Store in Vector Database:** The chunks and their corresponding embedding vectors are stored in a specialized **vector database** (like FAISS, Pinecone, or Chroma). This database is optimized for extremely fast similarity searches.

### Stage 2: Retrieval & Generation (Answering a User's Question)

1.  **Embed the Query:** The user's question is converted into an embedding vector using the *same* embedding model from the indexing stage.
2.  **Similarity Search:** The vector database searches for the `k` most similar document chunks to the user's query vector.
3.  **Augment the Prompt:** The retrieved chunks (the "context") are added to the user's original question in a prompt that is sent to the LLM.
4.  **Generate the Answer:** The LLM generates an answer based *only* on the provided context. This forces the model to ground its response in the retrieved facts, rather than relying on its internal (and potentially outdated) knowledge.

## 3. Hands-On: Building the Core Components

### A. Embeddings with `sentence-transformers`

Embeddings are the backbone of RAG. Let's see how to create them.



### B. Vector Database with `faiss`

FAISS (Facebook AI Similarity Search) is a popular, open-source library for efficient similarity search.



> **💡 Pro Tip:** The choice of embedding model is critical. A model fine-tuned on your specific domain (e.g., legal or medical text) will significantly outperform a general-purpose model.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# 1. Load a pre-trained model from Hugging Face
# This model is small, fast, and performs very well for its size.
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Our document chunks
document_chunks = [
    "The Eiffel Tower is located in Paris, France.",
    "The Great Wall of China is one of the seven wonders of the world.",
    "Photosynthesis is the process by which plants use sunlight to synthesize foods."
]

# 3. Generate embeddings
embeddings = model.encode(document_chunks)

# Each embedding is a vector (a list of numbers)
print("Shape of embeddings:", embeddings.shape)
print("Embedding for the first chunk:\n", embeddings[0][:10], "...") # Print first 10 dimensions

In [ ]:
import faiss

# Assume 'embeddings' is the output from the previous step
# FAISS requires numpy arrays
embeddings = np.array(embeddings, dtype='float32')

# 1. Build the FAISS index
dimension = embeddings.shape[1]  # The size of our embedding vector (e.g., 384 for MiniLM)
index = faiss.IndexFlatL2(dimension)
index.add(embeddings) # Add our document chunk embeddings to the index

# 2. Embed the user's query
user_query = "What is the process plants use to get energy?"
query_embedding = model.encode([user_query])

# 3. Perform the search
k = 1 # Retrieve the top 1 most similar chunk
distances, indices = index.search(np.array(query_embedding, dtype='float32'), k)

# 4. Get the result
retrieved_chunk_index = indices[0][0]
retrieved_chunk = document_chunks[retrieved_chunk_index]

print(f"User Query: {user_query}")
print(f"Most Relevant Document: '{retrieved_chunk}'")

## Practice Quizzes

Test your understanding of the core RAG concepts.

### Quiz 1: What is the primary function of a vector database in a RAG system?
- [ ] To store the Large Language Model itself.
- [ ] To store raw text documents in a compressed format.
- [✓] To store text chunks and their corresponding embedding vectors, enabling efficient similarity search.
- [ ] To generate text answers based on a user's query.

### Quiz 2: Why is "chunking" a necessary step in the RAG indexing pipeline?
- [ ] To make the documents harder to read for security purposes.
- [✓] To break down large documents into smaller, semantically coherent pieces that can fit within an LLM's limited context window.
- [ ] To increase the number of files stored in the vector database.
- [ ] To convert the documents into a different language.

### Quiz 3: During the retrieval step, how does the system determine which documents are "relevant" to the user's query?
- [ ] It performs a keyword search for the words in the query.
- [ ] It asks the LLM which documents seem most relevant.
- [✓] It calculates the vector similarity (e.g., cosine similarity or L2 distance) between the query embedding and all chunk embeddings in the database.
- [ ] It retrieves documents at random to ensure diversity.